# Wine Quality Prediction: Classification and Regression

**A supervised machine learning project using the UCI White Wine Quality dataset**

This notebook investigates whether physicochemical properties can be used to predict the sensory quality score of white wine. Two approaches are explored:

- **Classification:** predict the integer quality category directly.
- **Regression:** predict a continuous quality score, then round it to the nearest integer to compare its categorical predictions with the classifiers.

The project compares several commonly used machine learning algorithms, applies hyperparameter tuning with `GridSearchCV`, and evaluates model performance using accuracy, quadratic weighted kappa (QWK), and, for regression, \(R^2\).

**Notebook note:** The original modelling workflow, variable names, functions, parameter grids, and evaluation approach have been retained. The additions are explanatory headings and comments to make the analysis easier to follow.

## 1. Imports and setup

Import the data-analysis, preprocessing, model-selection, machine-learning, and evaluation tools used throughout the notebook.

In [72]:
# Import statements from the original notebook; imports are intentionally retained as written.
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from scipy import stats

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (OneHotEncoder, TargetEncoder, StandardScaler)
from sklearn.impute import SimpleImputer
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import TunedThresholdClassifierCV
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.dummy import DummyRegressor
from sklearn.metrics import roc_auc_score
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import r2_score
from sklearn.metrics import cohen_kappa_score
from sklearn.linear_model import Ridge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingClassifier

## 2. Load and inspect the dataset

In [39]:
# Read the UCI white-wine CSV file.
winedata_raw = pd.read_csv('wine+quality/winequality-white.csv', sep=';')

In [40]:
# Display a sample of the raw dataset.
winedata_raw.head(20)

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.0,0.27,0.36,20.70,0.045,45.0,170.0,1.0010,3.00,0.45,8.8,6
1,6.3,0.30,0.34,1.60,0.049,14.0,132.0,0.9940,3.30,0.49,9.5,6
2,8.1,0.28,0.40,6.90,0.050,30.0,97.0,0.9951,3.26,0.44,10.1,6
3,7.2,0.23,0.32,8.50,0.058,47.0,186.0,0.9956,3.19,0.40,9.9,6
4,7.2,0.23,0.32,8.50,0.058,47.0,186.0,0.9956,3.19,0.40,9.9,6
5,8.1,0.28,0.40,6.90,0.050,30.0,97.0,0.9951,3.26,0.44,10.1,6
6,6.2,0.32,0.16,7.00,0.045,30.0,136.0,0.9949,3.18,0.47,9.6,6
7,7.0,0.27,0.36,20.70,0.045,45.0,170.0,1.0010,3.00,0.45,8.8,6
8,6.3,0.30,0.34,1.60,0.049,14.0,132.0,0.9940,3.30,0.49,9.5,6
9,8.1,0.22,0.43,1.50,0.044,28.0,129.0,0.9938,3.22,0.45,11.0,6


In [41]:
# Define the feature matrix and target vector.
X = winedata_raw[['fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar', 'chlorides', 'free sulfur dioxide',
      'total sulfur dioxide', 'density', 'pH', 'sulphates', 'alcohol']]
y = winedata_raw['quality']

In [42]:
# Inspect the distribution of target quality scores.
y.value_counts()

quality
6    2198
5    1457
7     880
8     175
4     163
3      20
9       5
Name: count, dtype: int64

### Train/test split

In [45]:
# Create the stratified 80/20 training and test split.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state=0, stratify = y)

In [46]:
# Inspect training-set dimensions.
X_train.shape

(3918, 11)

In [47]:
# Inspect test-set dimensions.
X_test.shape

(980, 11)

## 3. Preprocessing and cross-validation

The numerical features are processed using a pipeline containing median imputation and standardization. Keeping preprocessing in a pipeline allows the transformations to be fitted within each cross-validation training fold.

In [48]:
# Create the shared numerical preprocessing pipeline.
num_pipeline = Pipeline([ ('imputer', SimpleImputer(strategy = 'median')), ('scaler', StandardScaler()) ])

In [49]:
# Define the cross-validation splitter used in the grid searches.
cv_s = StratifiedKFold(n_splits=4, shuffle=True, random_state = 0)

## 4. Evaluation functions

The following two functions evaluate the tuned models on the held-out test set. `fnl_tst()` is used for classification, while `fnl_tst_reg()` is used for regression. For regression, the continuous predictions are rounded to the nearest integer and clipped to the range 1–9 before calculating accuracy and QWK.

In [50]:
# Classification test-set evaluation function (original implementation retained).
def fnl_tst(model):
    

    y_pred = model.best_estimator_.predict(X_test)

    qwk = cohen_kappa_score(y_test, y_pred, weights="quadratic")

    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("Quadratic Weighted Kappa:", qwk)
    print("Precision:", precision_score(y_test, y_pred, average = 'macro'))
    print("Recall:", recall_score(y_test, y_pred, average = 'macro'))
    print("F1:", f1_score(y_test, y_pred, average = 'macro'))


In [51]:
# Regression test-set evaluation function (original implementation retained).
def fnl_tst_reg(model):

    y_pred = model.best_estimator_.predict(X_test)

    y_pred_rounded = np.rint(y_pred).astype(int)
    y_pred_rounded = np.clip(y_pred_rounded, 1, 9)

    qwk = cohen_kappa_score(y_test, y_pred_rounded, weights="quadratic")

    print("R^2 value:", r2_score(y_test, y_pred))
    print("Quadratic Weighted Kappa:", qwk)
    
    print("Accuracy:", accuracy_score(y_test, y_pred_rounded))


### Evaluation metric: quadratic weighted kappa

Quadratic weighted Cohen's kappa measures agreement between predicted and actual categories while accounting for the magnitude of disagreements. Because wine quality scores are ordered, a prediction that is one point away from the true score is penalized less than a prediction that is several points away.

## 5. Classification models

### 5.1 Logistic Regression

The first classification model is Logistic Regression. `GridSearchCV` is used to select the regularization parameter `C`, with mean cross-validation accuracy as the selection criterion.

In [52]:
# Logistic Regression grid-search function (original implementation retained).
def lgstc_grdsrch():

    clf_lgstc = Pipeline([ ('preprocessing', num_pipeline), ('model', LogisticRegression(max_iter = 1000)) ])
    
    params_lgstc = {'model__C':[0.000001,0.00001, 0.0001, 0.001, 0.01, 0.1, 1, 10]}

    grdsrch_lgstc = GridSearchCV(estimator = clf_lgstc, param_grid = params_lgstc, cv=cv_s, return_train_score=True, scoring="accuracy")

    grdsrch_lgstc.fit(X_train, y_train)

    print("Best parameter:", grdsrch_lgstc.best_params_)
    print("Best score:", grdsrch_lgstc.best_score_)

    return grdsrch_lgstc

In [53]:
# Run Logistic Regression grid search.
lgstc_grdsrch = lgstc_grdsrch()

Best parameter: {'model__C': 0.1}
Best score: 0.5318984386400117


After defining the search function, fit the model and use the fitted search object to evaluate predictions on the test set.

In [54]:
# Evaluate Logistic Regression on the test set.
fnl_tst(lgstc_grdsrch)

Accuracy: 0.5469387755102041
Quadratic Weighted Kappa: 0.43835823718554945
Precision: 0.383559820461229
Recall: 0.23495894140224038
F1: 0.2460801092793919


C:\Users\LENOVO\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


The original test-set result for Logistic Regression is approximately 0.54 accuracy and 0.43 QWK. The next model explored is Random Forest classification.

### 5.2 Random Forest Classifier

In [55]:
# Random Forest classification grid-search function (original implementation retained).
def rf_grdsrch():

    
    clf_lgstc = Pipeline([ ('preprocessing', num_pipeline), ('model', RandomForestClassifier(random_state = 0)) ])
    params = {'model__n_estimators': [8, 10, 12, 14, 16, 18], 'model__max_depth': [None, 10, 15, 20, 25]}
    
    grdsrch = GridSearchCV(estimator = clf_lgstc, param_grid = params, cv=cv_s, return_train_score=True, scoring="accuracy")

    grdsrch.fit(X_train, y_train)

    print("Best parameter:", grdsrch.best_params_)
    print("Best score:", grdsrch.best_score_)

    return grdsrch

In [56]:
# Run Random Forest classification grid search.
rf_grdsrch = rf_grdsrch()

Best parameter: {'model__max_depth': 25, 'model__n_estimators': 12}
Best score: 0.6467493381417939


In [57]:
# Evaluate Random Forest classification on the test set.
fnl_tst(rf_grdsrch)

Accuracy: 0.6622448979591836
Quadratic Weighted Kappa: 0.6452319710393446
Precision: 0.5184246721944593
Recall: 0.37977369954542267
F1: 0.41965959567865135


C:\Users\LENOVO\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


### 5.3 Support Vector Machine (RBF kernel)

In [58]:
# Support Vector Machine grid-search function (original implementation retained).
def SVM_grdsrch():

    
    clf_svm = Pipeline([ ('preprocessing', num_pipeline), ('model', SVC(kernel = 'rbf')) ])
    params = {'model__gamma': [0.01, 0.1, 1, 10,100]}

    
    grdsrch = GridSearchCV(estimator = clf_svm, param_grid = params, cv=cv_s, return_train_score=True, scoring="accuracy")

    grdsrch.fit(X_train, y_train)

    print("Best parameter:", grdsrch.best_params_)
    print("Best score:", grdsrch.best_score_)

    return grdsrch

In [59]:
# Run SVM grid search.
svm_grdsrch = SVM_grdsrch()

Best parameter: {'model__gamma': 1}
Best score: 0.6128158158053824


In [60]:
# Evaluate SVM on the test set.
fnl_tst(svm_grdsrch)

Accuracy: 0.6673469387755102
Quadratic Weighted Kappa: 0.5556136523448063
Precision: 0.5947151138812761
Recall: 0.32683942836741064
F1: 0.3624984035388725


C:\Users\LENOVO\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


### 5.4 Gradient Boosting Classifier

In [77]:
# Gradient Boosting classification grid-search function (original implementation retained).
def gb_grdsrch():

    
    clf_gb = Pipeline([ ('preprocessing', num_pipeline), ('model', GradientBoostingClassifier(max_depth = 3, random_state = 0)) ])
    params = {'model__learning_rate': [0.01, 0.1, 1.0, 10]}

    
    grdsrch = GridSearchCV(estimator = clf_gb, param_grid = params, cv=cv_s, return_train_score=True, scoring="accuracy")

    grdsrch.fit(X_train, y_train)

    print("Best parameter:", grdsrch.best_params_)
    print("Best score:", grdsrch.best_score_)

    return grdsrch

In [75]:
# Run Gradient Boosting grid search.
gb_grdsrch = gb_grdsrch()

Best parameter: {'model__learning_rate': 0.1}
Best score: 0.5819294469575368


In [76]:
# Evaluate Gradient Boosting classification on the test set.
fnl_tst(gb_grdsrch)

Accuracy: 0.5979591836734693
Quadratic Weighted Kappa: 0.5567285289628023
Precision: 0.44704755320933753
Recall: 0.3175819578065528
F1: 0.3514711067838964


C:\Users\LENOVO\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


## 6. Regression models

The second approach treats wine quality as a continuous target. The regression models predict a floating-point value. To compare their predictions with the classification models, the values are rounded to the nearest integer before calculating accuracy and QWK. The continuous predictions are also evaluated using \(R^2\).

The three regression algorithms considered are Ridge Regression, K-Nearest Neighbors Regression, and Random Forest Regression.

### 6.1 Ridge Regression

In [61]:
# Ridge Regression grid-search function (original implementation retained).
def ridgrgrsn_grdsrch():

    
    reg_ridge = Pipeline([ ('preprocessing', num_pipeline), ('model', Ridge(max_iter = 2000)) ])
    params = {'model__alpha': [0.01, 0.1, 1, 10, 100, 1000, 10000]}

    
    grdsrch = GridSearchCV(estimator = reg_ridge, param_grid = params, cv=cv_s, return_train_score=True, scoring="r2")

    grdsrch.fit(X_train, y_train)

    print("Best parameter:", grdsrch.best_params_)
    print("Best score:", grdsrch.best_score_)

    return grdsrch

In [62]:
# Run Ridge Regression grid search.
rdge_grdsrch = ridgrgrsn_grdsrch()

Best parameter: {'model__alpha': 100}
Best score: 0.26727412933877265


In [63]:
# Evaluate Ridge Regression on the test set.
fnl_tst_reg(rdge_grdsrch)

R^2 value: 0.3045470318189766
Quadratic Weighted Kappa: 0.4086486908034548
Accuracy: 0.5316326530612245


### 6.2 K-Nearest Neighbors Regression

In [64]:
# KNN Regression grid-search function (original implementation retained).
def knn_grdsrch():

    
    reg_knn = Pipeline([ ('preprocessing', num_pipeline), ('model', KNeighborsRegressor()) ])
    params = {'model__n_neighbors': [2,4,6,8,10,12,14,16,18]}

    
    grdsrch = GridSearchCV(estimator = reg_knn, param_grid = params, cv=cv_s, return_train_score=True, scoring="r2")

    grdsrch.fit(X_train, y_train)

    print("Best parameter:", grdsrch.best_params_)
    print("Best score:", grdsrch.best_score_)

    return grdsrch

In [65]:
# Run KNN Regression grid search.
knn_grdsrch = knn_grdsrch()

Best parameter: {'model__n_neighbors': 12}
Best score: 0.3273698681649857


In [66]:
# Evaluate KNN Regression on the test set.
fnl_tst_reg(knn_grdsrch)

R^2 value: 0.3924359053716281
Quadratic Weighted Kappa: 0.4908469920667907
Accuracy: 0.5418367346938775


### 6.3 Random Forest Regression

In [67]:
# Random Forest Regression grid-search function (original implementation retained).
def rfreg_grdsrch():

    
    reg_rf = Pipeline([ ('preprocessing', num_pipeline), ('model', RandomForestRegressor(random_state=0)) ])
    params = {'model__n_estimators': [8, 10, 12, 14, 16, 18, 20, 22, 24, 26], 'model__max_depth': [None, 10, 15, 20, 25]}

    
    grdsrch = GridSearchCV(estimator = reg_rf, param_grid = params, cv=cv_s, return_train_score=True, scoring="r2")

    grdsrch.fit(X_train, y_train)

    print("Best parameter:", grdsrch.best_params_)
    print("Best score:", grdsrch.best_score_)

    return grdsrch

In [68]:
# Run Random Forest Regression grid search.
rfreg_grdsrch = rfreg_grdsrch()

Best parameter: {'model__max_depth': None, 'model__n_estimators': 26}
Best score: 0.47297246516299835


In [78]:
# Evaluate Random Forest Regression on the test set.
fnl_tst_reg(rfreg_grdsrch)

R^2 value: 0.5291653533468622
Quadratic Weighted Kappa: 0.6605337876206702
Accuracy: 0.6724489795918367


## 7. Baseline models

The tuned models are benchmarked against a dummy classifier that always predicts the most frequent quality category and a dummy regressor that always predicts the mean quality score.

In [70]:
# Fit and evaluate the dummy classification and regression baselines.
dummy_reg = DummyRegressor(strategy="mean")
dummy_reg.fit(X_train, y_train)

dummy_clf = DummyClassifier(strategy = 'most_frequent').fit(X_train, y_train)

y_pred_dummy_clf = dummy_clf.predict(X_test)
y_pred_dummy_reg = dummy_reg.predict(X_test)

print("dummy r^2 value: ", round(r2_score(y_test, y_pred_dummy_reg),2))
print("dummy accuracy: ", round(accuracy_score(y_test, y_pred_dummy_clf),2))

dummy r^2 value:  -0.0
dummy accuracy:  0.45


## 8. Conclusion

In this experiment, Random Forest Regression gives the strongest overall results among the models tested. On the held-out test data, the recorded metrics are approximately:

- \(R^2 = 0.53\)
- Quadratic weighted kappa = 0.66
- Accuracy after rounding = 0.67

These results suggest that, for this particular experiment and train/test split, predicting a continuous quality score and rounding it gives a modest improvement over the tested classification models.

In [ ]:
# Original code cell.